# TFT Training — Cáceres NWP Forecast Model

This notebook trains a TFT on the NWP-augmented dataset produced by `6_nwp_data_assembly.ipynb`
and preprocessed by `3b_nwp_preprocessing.ipynb`.

## Key difference from notebook 4 (standard model)
The 7 Open-Meteo ECMWF forecast columns are added to `time_varying_known_reals`, making them
**decoder-visible**. The TFT can now attend to tomorrow's predicted weather when generating
the 24h forecast. ERA5 observed weather stays in `time_varying_unknown_reals` (encoder-only).

| | Standard model (nb 4) | NWP model (this notebook) |
|---|---|---|
| `known_future` (decoder) | 9 features (calendar + geometry) | 16 features (+7 NWP forecast) |
| `observed` (encoder only) | 8 ERA5 features | 8 ERA5 features (unchanged) |

**Pipeline steps:**

| # | Step | Purpose |
|---|---|---|
| 0 | Imports & setup | Libraries, GPU detection, random seed |
| 1 | Configuration | Sequence lengths, feature lists, denorm params |
| 2 | Data loading | Load NWP CSVs, add time_idx and group_id |
| 3 | TimeSeriesDataSet construction | 16 known_future + 8 observed |
| 4 | Learning rate finder | Establish LR range before tuning |
| 5 | Hyperparameter tuning | Optuna TPE, 15 trials |
| 6 | Final model training | Best HPs, early stopping, checkpointing |
| 7 | Validation evaluation | Denormalized metrics |
| 8 | Save artifacts | Model, study, metrics |

## Step 0 — Imports & setup

In [ ]:
import os, json, warnings, pickle
import numpy as np
import numpy._core.multiarray
import pandas as pd
import matplotlib.pyplot as plt

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.tuner import Tuner

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss, MAE, RMSE
from pytorch_forecasting.data.encoders import TorchNormalizer

import optuna
from optuna.samplers import TPESampler

import functools
_orig_torch_load = torch.load
@functools.wraps(_orig_torch_load)
def _patched_torch_load(*args, **kwargs):
    if kwargs.get('weights_only') is None:
        kwargs['weights_only'] = False
    return _orig_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pl.seed_everything(42)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
%pip install torchmetrics

## Step 1 — Configuration

In [ ]:
# ── Paths ──
DATA_DIR  = os.path.join(os.getcwd(), 'data')
MODEL_DIR = os.path.join(os.getcwd(), 'models', 'nwp')
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Sequence lengths ──
MAX_ENCODER_LENGTH    = 168   # 7 days of hourly history
MAX_PREDICTION_LENGTH = 24    # 1-day-ahead forecast

# ── Feature classification (matches 3b_nwp_preprocessing.ipynb Step 8) ──
KNOWN_FUTURE = [
    # calendar + solar geometry (deterministic)
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'doy_sin', 'doy_cos', 'solar_zenith', 'solar_azimuth', 'clearsky_ghi',
    # NWP forecast weather — decoder-visible (NEW vs standard model)
    'ssrd_wm2_forecast', 'temperature_2m_C_forecast', 'dewpoint_2m_C_forecast',
    'surface_pressure_hPa_forecast', 'total_precip_mm_forecast',
    'kt_forecast', 'dewpoint_depression_C_forecast',
]
OBSERVED = [
    # ERA5 observed weather — encoder-only (unchanged)
    'dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa',
    'total_precip_mm', 'ssrd_wm2', 'strd_wm2', 'kt', 'dewpoint_depression_C',
]
TARGET = 'pv_generation_mwh'

# ── Denormalization parameters ──
with open(os.path.join(DATA_DIR, 'nwp_preprocessing_params.json')) as f:
    pp = json.load(f)
TARGET_MEAN = pp['target_mean']
TARGET_STD  = pp['target_std']

print(f"Encoder: {MAX_ENCODER_LENGTH}h ({MAX_ENCODER_LENGTH//24}d)")
print(f"Prediction: {MAX_PREDICTION_LENGTH}h")
print(f"Known future features: {len(KNOWN_FUTURE)} (9 calendar/geometry + 7 NWP forecast)")
print(f"Observed features: {len(OBSERVED)} (ERA5 only)")
print(f"Target denorm: z * {TARGET_STD:.4f} + {TARGET_MEAN:.4f}")

## Step 2 — Load data & prepare for TimeSeriesDataSet

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'nwp_train_processed.csv'),
                        parse_dates=['datetime_utc'], index_col='datetime_utc')
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'nwp_val_processed.csv'),
                        parse_dates=['datetime_utc'], index_col='datetime_utc')

print(f"Train: {len(train_df):,} rows  {train_df.index[0]} -> {train_df.index[-1]}")
print(f"Val:   {len(val_df):,} rows   {val_df.index[0]} -> {val_df.index[-1]}")
print(f"Train columns ({train_df.shape[1]}): {list(train_df.columns)}")

assert train_df.isna().sum().sum() == 0, "NaN in training data"
assert val_df.isna().sum().sum() == 0, "NaN in validation data"

train_df = train_df.reset_index()
val_df   = val_df.reset_index()

train_df['time_idx'] = np.arange(len(train_df))
val_df['time_idx']   = np.arange(len(train_df), len(train_df) + len(val_df))

train_df['group_id'] = '0'
val_df['group_id']   = '0'

assert train_df['time_idx'].iloc[-1] + 1 == val_df['time_idx'].iloc[0], \
    "time_idx not continuous across train/val boundary"

print(f"\ntime_idx range — train: [0, {train_df['time_idx'].max()}], "
      f"val: [{val_df['time_idx'].min()}, {val_df['time_idx'].max()}]")

## Step 3 — Construct TimeSeriesDataSet

In [ ]:
training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx='time_idx',
    target=TARGET,
    group_ids=['group_id'],

    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    min_encoder_length=MAX_ENCODER_LENGTH,

    time_varying_known_reals=KNOWN_FUTURE,
    time_varying_unknown_reals=OBSERVED,

    target_normalizer=None,
    scalers={col: None for col in KNOWN_FUTURE + OBSERVED},

    add_relative_time_idx=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
)

val_with_context = pd.concat([
    train_df.iloc[-MAX_ENCODER_LENGTH:],
    val_df
], ignore_index=True)

val_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    val_with_context,
    stop_randomization=True,
)

BATCH_SIZE = 64
train_dataloader = training_dataset.to_dataloader(train=True,  batch_size=BATCH_SIZE, num_workers=2)
val_dataloader   = val_dataset.to_dataloader(     train=False, batch_size=BATCH_SIZE, num_workers=2)

print(f"Training dataset: {len(training_dataset)} samples")
print(f"Validation dataset: {len(val_dataset)} samples")

x, y = next(iter(train_dataloader))
print(f"\nBatch shapes:")
print(f"  Encoder cont. inputs: {x['encoder_cont'].shape}")
print(f"  Decoder cont. inputs: {x['decoder_cont'].shape}  ← includes {len(KNOWN_FUTURE)} known_future")
print(f"  Target:               {y[0].shape}")

## Step 4 — Learning rate finder

In [ ]:
tft_tmp = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.001,
    hidden_size=32,
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=QuantileLoss(),
    optimizer='adam',
)

trainer_tmp = pl.Trainer(accelerator='auto', gradient_clip_val=0.1, max_epochs=3)
tuner = Tuner(trainer_tmp)
lr_finder = tuner.lr_find(
    tft_tmp,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
    min_lr=1e-6, max_lr=1.0, num_training=100,
)

fig = lr_finder.plot(suggest=True)
plt.title("Learning Rate Finder — NWP Model")
plt.show()

suggested_lr = lr_finder.suggestion()
print(f"Suggested LR: {suggested_lr:.6f}")

del tft_tmp, trainer_tmp
torch.cuda.empty_cache()

## Step 5 — Hyperparameter tuning (Optuna, TPE, 15 trials)

In [ ]:
def optuna_objective(trial):
    hidden_size            = trial.suggest_categorical('hidden_size', [16, 32, 64, 128])
    lstm_layers            = trial.suggest_int('lstm_layers', 1, 2)
    attention_head_size    = trial.suggest_categorical('attention_head_size', [1, 2, 4])
    dropout                = trial.suggest_float('dropout', 0.1, 0.4)
    hidden_continuous_size = trial.suggest_categorical('hidden_continuous_size', [8, 16, 32])
    learning_rate          = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    tft = TemporalFusionTransformer.from_dataset(
        training_dataset,
        hidden_size=hidden_size,
        lstm_layers=lstm_layers,
        attention_head_size=attention_head_size,
        dropout=dropout,
        hidden_continuous_size=hidden_continuous_size,
        learning_rate=learning_rate,
        loss=QuantileLoss(),
        optimizer='adam',
        reduce_on_plateau_patience=3,
        output_size=7,
    )

    trainer = pl.Trainer(
        max_epochs=10,
        accelerator='auto',
        gradient_clip_val=0.1,
        callbacks=[EarlyStopping(monitor='val_loss', patience=3, mode='min', verbose=False)],
        enable_model_summary=False,
        enable_progress_bar=False,
        enable_checkpointing=False,
        logger=False,
    )

    try:
        trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
        val_loss = trainer.callback_metrics['val_loss'].item()
    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        val_loss = float('inf')
    finally:
        torch.cuda.empty_cache()

    return val_loss

In [ ]:
study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=42),
    study_name='tft_caceres_nwp_hp_tuning',
)

study.optimize(optuna_objective, n_trials=15, show_progress_bar=True, gc_after_trial=True)

print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best val_loss: {study.best_value:.6f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

trial_numbers = [t.number for t in study.trials if t.value is not None and t.value < float('inf')]
trial_values  = [t.value  for t in study.trials if t.value is not None and t.value < float('inf')]
best_so_far   = np.minimum.accumulate(trial_values)

axes[0].scatter(trial_numbers, trial_values, alpha=0.6, label='Trial value')
axes[0].plot(trial_numbers, best_so_far, 'r-', linewidth=2, label='Best so far')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Validation Loss')
axes[0].set_title('Optimization History — NWP Model')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

importances = optuna.importance.get_param_importances(study)
axes[1].barh(list(importances.keys()), list(importances.values()))
axes[1].set_xlabel('Importance')
axes[1].set_title('Hyperparameter Importances')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6 — Train final model with best HPs

In [ ]:
best = study.best_params

tft_final = TemporalFusionTransformer.from_dataset(
    training_dataset,
    hidden_size=best['hidden_size'],
    lstm_layers=best['lstm_layers'],
    attention_head_size=best['attention_head_size'],
    dropout=best['dropout'],
    hidden_continuous_size=best['hidden_continuous_size'],
    learning_rate=best['learning_rate'],
    loss=QuantileLoss(),
    optimizer='adam',
    reduce_on_plateau_patience=5,
    output_size=7,
)

print(f"Model parameters: {sum(p.numel() for p in tft_final.parameters()):,}")
print(f"Best HPs: {best}")

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=MODEL_DIR,
    filename='tft_nwp_best',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
)

trainer_final = pl.Trainer(
    max_epochs=75,
    accelerator='auto',
    gradient_clip_val=0.1,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, mode='min', verbose=True),
        checkpoint_callback,
        LearningRateMonitor(logging_interval='epoch'),
    ],
    log_every_n_steps=50,
)

trainer_final.fit(tft_final, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

print(f"\nBest model saved to: {checkpoint_callback.best_model_path}")
print(f"Best val_loss: {checkpoint_callback.best_model_score:.6f}")

## Step 7 — Validation evaluation

In [ ]:
def denormalize(z_values):
    return z_values * TARGET_STD + TARGET_MEAN

def compute_metrics(actual_mwh, predicted_mwh):
    residuals = actual_mwh - predicted_mwh
    mae  = np.abs(residuals).mean()
    rmse = np.sqrt((residuals ** 2).mean())
    ss_res = (residuals ** 2).sum()
    ss_tot = ((actual_mwh - actual_mwh.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot
    return {'MAE_MWh': float(mae), 'RMSE_MWh': float(rmse), 'R2': float(r2)}

In [ ]:
best_model = TemporalFusionTransformer.load_from_checkpoint(checkpoint_callback.best_model_path)
best_model.eval()

val_predictions = best_model.predict(
    val_dataloader, mode='prediction', return_x=True, return_y=True,
)

pred_z   = val_predictions.output.cpu().numpy()
actual_z = val_predictions.y[0].cpu().numpy()

pred_mwh   = np.clip(denormalize(pred_z.flatten()), 0, None)
actual_mwh = denormalize(actual_z.flatten())

val_metrics = compute_metrics(actual_mwh, pred_mwh)
print("Validation Metrics (original MWh scale):")
for k, v in val_metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"\nVal quantile loss (normalized): {checkpoint_callback.best_model_score:.6f}")

## Step 8 — Save artifacts

In [ ]:
with open(os.path.join(MODEL_DIR, 'nwp_hp_study.pkl'), 'wb') as f:
    pickle.dump(study, f)

hp_results = {
    'model': 'nwp',
    'best_trial': study.best_trial.number,
    'best_val_loss': study.best_value,
    'best_params': study.best_params,
    'n_trials': len(study.trials),
    'val_metrics_mwh': val_metrics,
    'feature_config': {
        'known_future': KNOWN_FUTURE,
        'observed': OBSERVED,
        'n_known_future': len(KNOWN_FUTURE),
        'n_observed': len(OBSERVED),
    },
    'sequence_config': {
        'max_encoder_length': MAX_ENCODER_LENGTH,
        'max_prediction_length': MAX_PREDICTION_LENGTH,
    },
}

with open(os.path.join(MODEL_DIR, 'nwp_hp_study_results.json'), 'w') as f:
    json.dump(hp_results, f, indent=2)

val_pred_df = pd.DataFrame({
    'actual_z': actual_z.flatten(),
    'predicted_z': pred_z.flatten(),
    'actual_mwh': actual_mwh,
    'predicted_mwh': pred_mwh,
})
val_pred_df.to_csv(os.path.join(MODEL_DIR, 'nwp_val_predictions.csv'), index=False)

print("Artifacts saved:")
for f_name in sorted(os.listdir(MODEL_DIR)):
    f_path = os.path.join(MODEL_DIR, f_name)
    if os.path.isfile(f_path):
        print(f"  {f_name}: {os.path.getsize(f_path)/1e6:.1f} MB")

print("\nNext step: run 5c_nwp_evaluation.ipynb")